# Aquisição do histórico USD/BRL (Yahoo Finance)

Baixa o ticker `BRL=X` em frequência diária e grava um CSV bruto. O limite final é o dia atual em São Paulo (exclusivo), para evitar uma cotação ainda em formação. Valores ausentes são preservados para avaliação na EDA.

Dependências: Python 3.9+ com `zoneinfo`, `pandas` e `requests`. Execute a partir da raiz do repositório ou da pasta `notebooks/`.

In [ ]:
from datetime import date, datetime, time, timezone, timedelta
from pathlib import Path
from zoneinfo import ZoneInfo

import pandas as pd
import requests

SYMBOL = 'BRL=X'  # BRL por 1 USD
START = date(2023, 1, 1)
END_EXCLUSIVE = datetime.now(ZoneInfo('America/Sao_Paulo')).date()
cwd = Path.cwd()
if cwd.name == 'notebooks' and (cwd.parent / 'README.md').exists():
    repo_root = cwd.parent
elif (cwd / 'notebooks').is_dir() and (cwd / 'README.md').exists():
    repo_root = cwd
else:
    raise RuntimeError('Execute a partir da raiz do repositório ou da pasta notebooks/')
OUTPUT = repo_root / 'data' / 'raw' / 'usd_brl_daily.csv'

def unix_midnight(day):
    return int(datetime.combine(day, time.min, timezone.utc).timestamp())

url = f'https://query2.finance.yahoo.com/v8/finance/chart/{SYMBOL}'
response = requests.get(
    url,
    params={
        'period1': unix_midnight(START),
        'period2': unix_midnight(END_EXCLUSIVE),
        'interval': '1d',
    },
    headers={'User-Agent': 'Mozilla/5.0'},
    timeout=30,
)
response.raise_for_status()
chart = response.json()['chart']
if chart['error'] is not None or not chart['result']:
    raise ValueError(f'Yahoo não retornou a série: {chart["error"]}')
result = chart['result'][0]
quote = result['indicators']['quote'][0]
timestamps = result['timestamp']
if any(len(quote[column]) != len(timestamps) for column in ('open', 'high', 'low', 'close', 'volume')):
    raise ValueError('Comprimentos inconsistentes na resposta do Yahoo')

exchange_tz = result['meta']['exchangeTimezoneName']
dates = pd.to_datetime(timestamps, unit='s', utc=True).tz_convert(exchange_tz).date
data = pd.DataFrame({
    'Date': dates,
    'Open': quote['open'],
    'High': quote['high'],
    'Low': quote['low'],
    'Close': quote['close'],
    'Volume': quote['volume'],
})
data = data.loc[(data['Date'] >= START) & (data['Date'] < END_EXCLUSIVE)]
data = data.sort_values('Date').reset_index(drop=True)
if data.empty or data['Date'].duplicated().any():
    raise ValueError('Série vazia ou com datas duplicadas; CSV não gravado')

OUTPUT.parent.mkdir(parents=True, exist_ok=True)
data.to_csv(OUTPUT, index=False, date_format='%Y-%m-%d')
saved = pd.read_csv(OUTPUT, parse_dates=['Date'])
assert len(saved) == len(data) and saved['Date'].is_monotonic_increasing
assert saved['Date'].is_unique
assert saved['Date'].min().date() == data['Date'].min()
assert saved['Date'].max().date() == data['Date'].max()
assert saved['Date'].min().date() >= START
assert saved['Date'].max().date() < END_EXCLUSIVE
assert saved['Date'].max() - saved['Date'].min() > timedelta(days=730)
assert saved['Close'].notna().any() and saved['Close'].dropna().gt(0).all()
print(f'Snapshot: {END_EXCLUSIVE}; {len(saved)} linhas: {saved.Date.min().date()} a {saved.Date.max().date()} -> {OUTPUT}')
print(f'Fechamentos ausentes preservados: {saved.Close.isna().sum()}')
